### Day 08  OOP 组合+Pythonic (L4)

今天让自定义对象 像内置类型一样自然。
你会学会:组合优于继承、运算符重载、迭代协议。

> 叙事锚点:电商 v2 最后一公里 -- 对象交互协议
> 整合 L1-L4:Product / Cart / Payment / Order 全系统

> 组合优于继承 (Composition over Inheritance)

**痛点**:购物车和订单,该用继承还是组合?

**类比**:
- is-a (继承): Dog is-a Animal
- has-a (组合): Order has-a Cart + has-a Payment

**解释**:如果两个类之间是 包含/使用 关系,不要用继承。
继承层级 > 3 层 = 设计坏味道。

In [ ]:
class ShoppingCart:
    def __init__(self, items):
        self.items = items
    def total(self):
        return sum(item[1] for item in self.items)
    def __len__(self):
        return len(self.items)

class Address:
    def __init__(self, city, detail):
        self.city = city
        self.detail = detail

class Order:
    def __init__(self, cart, payment, address):
        self.cart = cart
        self.payment = payment
        self.address = address
    def summary(self):
        return f"订单: {len(self.cart)} 件,总价: {self.cart.total()},地址: {self.address.city}"

# 组合:Order has-a 多个对象
items = [("Python入门", 50), ("算法图解", 30)]
cart = ShoppingCart(items)
addr = Address("北京", "朝阳")
order = Order(cart, None, addr)
print(order.summary())

> 逐行解剖
- ShoppingCart / Address 独立类
- Order 持有它们的实例,不继承
- 修改 Cart 不影响 Order,可自由替换

> BREAK IT:用继承模拟 has-a (反模式)

下面代码 能跑,但设计上是错的。请找出 3 个坏味道。

    class Order(ShoppingCart):  # Order is-a Cart? 错!
        def __init__(self):
            super().__init__([])
            self.payment = None

坏味道:
- Order 是 Cart 的子类 - 语义不对
- 修改 Cart 会无意中影响 Order
- Order 无法复用其他 Cart 实现

**练习**:组合设计

定义 Order 类,组合 Cart + Address。实现 summary()。

问自己:
- Order 应该继承 Cart 吗?
- 组合的好处是什么?

In [ ]:
# ============ 学员代码区 ============
class Address:
    pass

class Order:
    pass

# cart = ShoppingCart([...])
# order = Order(cart, addr)
# print(order.summary())
pass

# ============ 参考答案 ============
class ShoppingCart:
    def __init__(self, items): self.items = items
    def total(self): return sum(p[1] for p in self.items)
    def __len__(self): return len(self.items)

class Address:
    def __init__(self, city): self.city = city

class Order:
    def __init__(self, cart, address):
        self.cart = cart
        self.address = address
    def summary(self):
        return f"{len(self.cart)} 件,总价 {self.cart.total()},送到 {self.address.city}"

> 运算符重载: __add__

**痛点**: cart1 + cart2 会 TypeError。

**解释**: a + b 自动调用 a.__add__(b)。应返回 新对象,不修改原对象。

In [ ]:
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price
    def __eq__(self, other):
        return isinstance(other, Product) and self.name == other.name
    def __repr__(self):
        return f"Product({self.name}, {self.price})"

class ShoppingCart:
    def __init__(self, items=None):
        self.items = items or []
    def __add__(self, other):
        return ShoppingCart(self.items + other.items)
    def __len__(self):
        return len(self.items)
    def __iter__(self):
        return iter(self.items)
    def total(self):
        return sum(p.price for p in self.items)

cart1 = ShoppingCart([Product("A", 50), Product("B", 30)])
cart2 = ShoppingCart([Product("C", 20)])
cart3 = cart1 + cart2  # __add__
print(f"合并后: {len(cart3)} 件,总价: {cart3.total()}")
print(f"cart1 仍 {len(cart1)} 件")  # 未修改!

> 逐行解剖
- __add__ 返回新 ShoppingCart
- cart1 本身不变(安全语义)
- __len__ / __iter__ 让 len() 和 for 可用

> BREAK IT:只有 __len__ 没有 __iter__

类写了 __len__,len() 能跑。但 for x in obj 仍然报错!

In [ ]:
class BadCart:
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    # 没有 __iter__!

print(f"len 能跑: {len(BadCart([1, 2, 3]))}")

try:
    for item in BadCart([1, 2, 3]):
        print(item)
except TypeError as e:
    print(f"遍历报错: {e}")
    print("没有 __iter__,for 不工作!")

> 协议总结:len -> __len__; for -> __iter__; obj[key] -> __getitem__; print -> __str__

In [ ]:
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price
    def __repr__(self):
        return f"{self.name}:{self.price}"

class ShoppingCart:
    def __init__(self, items=None): self.items = items or []
    def __len__(self): return len(self.items)
    def __iter__(self):
        for item in self.items:
            yield item  # yield 自动实现迭代协议

cart = ShoppingCart([Product("A", 50), Product("B", 30)])
total = 0
for p in cart:  # 能用!
    total += p.price
print(f"遍历总价: {total}")  # 80

> __eq__ 同款判定

**痛点**:两对象属性相同,== 仍返回 False(默认 == 比较 is)。

**解释**:__eq__ 定义'相似'标准,常用于去重。

In [ ]:
class Product:
    def __init__(self, sku, name, price):
        self.sku = sku
        self.name = name
        self.price = price
    def __eq__(self, other):
        return isinstance(other, Product) and self.sku == other.sku
    def __hash__(self):
        return hash(self.sku)
    def __repr__(self):
        return f"[{self.sku}] {self.name}"

p1 = Product("A001", "Py", 50)
p2 = Product("A001", "Py", 50)  # 同款
p3 = Product("A002", "Algo", 30)
print(p1 == p2)  # True
print(p1 == p3)  # False
s = {p1, p2, p3}
print(f"去重后 {len(s)} 件")  # 2

**练习**:同名同款
定义 Product,__eq__ 比较 name。
验证两同名商品 == 为 True。
问: 还要实现什么才能放入 set?

In [ ]:
# ============ 学员代码区 ============
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price
    def __eq__(self, other):
        pass

# p1 = Product("A", 50); p2 = Product("A", 30)
# print(p1 == p2)
pass

# ============ 参考答案 ============
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price
    def __eq__(self, other):
        return isinstance(other, Product) and self.name == other.name
    def __hash__(self):
        return hash(self.name)

> __repr__ vs __str__

__str__:面向用户,可读。print 时触发。
__repr__:面向开发者,明确。提示符默认显示。

规则:总是实现 __repr__。__str__ 可选,未定义 fallback 到 __repr__。

In [ ]:
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price
    def __repr__(self):
        return f"Product({self.name!r}, {self.price})"
    def __str__(self):
        return f"[{self.name}] {self.price} 元"

p = Product("Python", 59.8)
print(repr(p))  # Product('Python', 59.8)
print(str(p))   # [Python] 59.8 元
print(p)        # [Python] 59.8 元 (__str__)

> 综合练习:电商订单系统 v2 (L1-L4 全员整合)

整合全部知识,写一个可运行的系统。

要求:
- L1:Product(@property + __str__ + 类属性 tax_rate)
- L2:PhysicalProduct/DigitalProduct 子类
- L3:Payment(abc.ABC) + Alipay/WeChatPay/ApplePay + checkout() 无 if-elif
- L4:Order 组合 Cart + Payment + Address
- L4:__add__/__len__/__iter__/__eq__

In [ ]:
import abc

# ============ 学员代码区 ============
class Product:
    pass

class Payment(abc.ABC):
    pass

class ShoppingCart:
    pass

class Order:
    pass

def checkout(cart_total, payment):
    pass

# cart1 = ShoppingCart([Product(...)])
# cart2 = ShoppingCart([Product(...)])
# cart3 = cart1 + cart2
# order = Order(cart3, Alipay(), Address(...))
# print(order.summary())
# checkout(80, Alipay())
pass

# ============ 参考答案(节选) ============
import abc

class Product:
    tax_rate = 0.13
    def __init__(self, name, price):
        self.name = name
        self.price = price
    @property
    def price(self):
        return self._price
    @price.setter
    def price(self, value):
        if value < 0: raise ValueError
        self._price = value
    def __eq__(self, other):
        return isinstance(other, Product) and self.name == other.name
    def __hash__(self):
        return hash(self.name)
    def __repr__(self):
        return f"Product({self.name!r}, {self.price})"
    def __str__(self):
        return f"[{self.name}] {self.price} 元"

class Payment(abc.ABC):
    @abc.abstractmethod
    def execute(self, amount): ...

class Alipay(Payment):
    def execute(self, amount):
        print(f"支付宝 {amount}"); return True

class WeChatPay(Payment):
    def execute(self, amount):
        print(f"微信 {amount}"); return True

class ApplePay(Payment):
    def execute(self, amount):
        print(f"Apple Pay {amount}"); return True

class ShoppingCart:
    def __init__(self, items=None): self.items = items or []
    def __add__(self, other):
        return ShoppingCart(self.items + other.items)
    def __len__(self): return len(self.items)
    def __iter__(self): yield from self.items
    def __repr__(self): return f"Cart({len(self.items)})"

class Address:
    def __init__(self, city): self.city = city

class Order:
    def __init__(self, cart, payment, address):
        self.cart, self.payment, self.address = cart, payment, address
    def summary(self): return f"{len(self.cart)} 件 -> {self.address.city}"

def checkout(total, payment):
    if total <= 0: return False
    return payment.execute(total)

p1 = Product("Python", 50); p2 = Product("Algo", 30)
cart = ShoppingCart([p1, p2])
order = Order(cart, Alipay(), Address("北京"))
print(order.summary())
checkout(80, Alipay())

> 今日小结

| 概念 | 解决的痛点 |
| --- | --- |
| 组合优于继承 | has-a 不用继承 |
| add / eq / lt | 自定义对象支持运算符 |
| len / iter | 支持 len / for |
| repr / str | 面向开发者/用户的两种打印 |

更多练习:
- 当堂练:course/lesson08/in_class/practice01-06.py
- 课后作业:course/lesson08/homework/task01-03.py